# SR Tutorial

2026 IAIFI Summer School

Tutorial lead: Jose M Munoz Arias

Professor: Miles Cranmer

# Airfoil Self-Noise

The first notebook was a set of problems where we already knew the answer, which is the only
honest way to check whether a prior helped. 

Now, we will spend a bit of time on a problem where we do not know the answer for.

It is a real dataset,
measured in a real wind tunnel, and the rest of the session is yours to poke at.

For the next half hour: we load the data, fit a boosted-tree model to have an okay baseline, and then make three attempts at an equation.
---

Julia takes a while to wake up, so let's start that now and read while it happens.

In [ ]:
# Uncomment on Colab. Locally you should already have these from this morning.
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-q",
#                 "pysr", "xgboost", "scikit-learn"], check=True)

In [ ]:
import os
# Julia fixes its thread count at startup, so this has to happen before `import pysr`.
os.environ.setdefault("PYTHON_JULIACALL_THREADS", "auto")
os.environ.setdefault("PYTHON_JULIACALL_AUTOLOAD_IPYTHON_EXTENSION", "no")

import time

import numpy as np
import pandas as pd

t0 = time.time()
import pysr
from pysr import PySRRegressor
print(f"pysr {pysr.__version__} imported in {time.time() - t0:.0f}s")

import session2_data as s2
from session2_data import (load_airfoil, split, view, r2, explained_by,
                           front_scores, best_on_front, shortest_above)

# Same determinism switch as this morning. Drop it once you start exploring and the
# searches get noticeably faster, at the cost of not reproducing exactly.
REPRO = dict(deterministic=True, parallelism="serial", verbosity=0, progress=False)

---

## 1. What was measured

NASA put a series of NACA 0012 blade sections into an anechoic wind tunnel, ran them at a
range of speeds and angles of attack, and recorded the noise each blade made *by itself* — no
engine, no gearbox, just a wing section and the air coming off its own trailing edge. Brooks,
Pope and Marcolini published it as NASA RP-1218 in 1989, and it has been on UCI ever since.

Each row is one one-third-octave frequency band of one run. Five numbers set the condition and
the sixth is what the microphones heard.

In [ ]:
df = load_airfoil()

for c, (what, unit) in s2.MEANING.items():
    print(f"  {c:6s} {what:58s} [{unit}]")
print()
df.head()

In [ ]:
df[["freq", "aoa", "chord", "U", "delta", "spl"]].describe().T

Two things to notice in that table before we fit anything. Frequency runs from 200 Hz to
20 kHz, a factor of a hundred, and the displacement thickness covers a factor of a hundred and
fifty. The target does not: the whole dataset lives inside a 38 dB window. So four of the five
inputs are spread over decades while the output is not.

And the wind speed only ever takes four values, the chord only six. This was a designed
experiment, not a log of whatever the weather happened to do.

In [ ]:
s2.plot_overview(df);

No single panel shows a curve. Each one is a band, because the other four columns are moving
underneath it. That is what makes this a fitting problem rather than a plotting problem.

---

## 2. A number worth chasing

Before trying to write an equation it is worth knowing what a model that is not trying to be
readable can do with these rows. Boosted trees are the obvious thing to ask.

Everything from here on is scored the same way: fit on three quarters of the rows, report
$R^2$ on the quarter that was held back.

In [ ]:
from xgboost import XGBRegressor

train, test = split(df)
y = df["spl"].values

X_all, _ = view(df, "raw")
trees = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.06).fit(X_all[train], y[train])

BASELINE = r2(trees.predict(X_all[test]), y[test])
print(f"400 boosted trees:   held-out R^2 = {BASELINE:.3f}")

That is the number. It is a good one, and nothing we write today in a form you can read out
loud is going to beat it outright — there is real acoustics in here that no short expression
captures. The question for the next twenty minutes is how close a *legible* model can get, and
what we have to know about the physics to get there.

---

## 3. Attempt one: the columns exactly as measured

The obvious first move. Five columns in, sound pressure level out, a reasonable set of
operators, and no opinion at all about what any of it means.

In [ ]:
SEARCH = dict(
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "exp", "square"],
    maxsize=25,          # largest expression the search may build
    niterations=40,      # how long the populations get to evolve
    random_state=0,
    **REPRO,
)
# Each of the three fits below takes roughly a minute on a laptop that is not busy
# doing something else. If yours is, drop niterations to 20 — the story survives it,
# the equations just come out a little shorter.

X_raw, names_raw = view(df, "raw")

t = time.time()
raw_model = PySRRegressor(**SEARCH)
raw_model.fit(X_raw[train], y[train], variable_names=names_raw)
print(f"{time.time() - t:.0f}s")

raw_front = front_scores(raw_model, X_raw[test], y[test])
raw_front

That table is the Pareto front with one column added. `loss` is what the search minimised, so
it can only fall as complexity rises. `test_r2` is those same equations scored on the quarter
of the rows they never saw, and *that* one is free to go back down — which is how you catch an
equation that has started memorising instead of explaining.

Now look at where the budget went. Read the front from the top and find the first equation
that uses a logarithm at all.

In [ ]:
raw_front[raw_front["equation"].str.contains("log")].head(3)[["complexity", "test_r2", "equation"]]

Nobody asked it to do that. Handed frequency in hertz and thickness in metres, the first
structural thing the search builds is the logarithm of their product — and then it squares it.
It is spending its opening nodes rediscovering that this problem lives on a log scale, and that
frequency and thickness belong together, before it can spend a single node on anything else.

That is the cost of handing over raw columns. Not that the search is weak, but that we made it
buy something we already knew.

In [ ]:
best_raw = best_on_front(raw_front)
print(best_raw["equation"])
print(f"\ncomplexity {best_raw['complexity']}    held-out R^2 = {best_raw['test_r2']:.3f}"
      f"    (trees: {BASELINE:.3f})")

---

## 4. Attempt two: put the columns on the scale they live on

So let's stop making it pay for that. Decibels are already a logarithmic quantity, and
frequency, chord, velocity and thickness are all strictly positive and span decades. Take logs
of the inputs, leave the target alone.

This is the move you would make on any dataset, without thinking very hard about this one.
Same operators, same budget, same rows.

In [ ]:
X_log, names_log = view(df, "log")
print("features:", names_log)

t = time.time()
log_model = PySRRegressor(**SEARCH)
log_model.fit(X_log[train], y[train], variable_names=names_log)
print(f"{time.time() - t:.0f}s")

log_front = front_scores(log_model, X_log[test], y[test])
best_log = best_on_front(log_front)
print()
print(best_log["equation"])
print(f"\ncomplexity {best_log['complexity']}    held-out R^2 = {best_log['test_r2']:.3f}"
      f"    (was {best_raw['test_r2']:.3f})")

Which is... barely anything. A couple of points of $R^2$ for a change that felt like it should
have mattered.

This is worth sitting with for a second, because it is the more common experience than the one
in the next section. The rescaling was not wrong — it is a perfectly sensible thing to have
done, and it made the equations shorter to write down. It just was not where the difficulty
was. The search had already worked around the scale problem on its own; all we did was refund
it a few nodes.

Getting a real improvement is going to take knowing something about trailing edges.

---

## 5. Attempt three: the right variable

Here is the one piece of physics in this notebook, and it is the piece a room full of
physicists will get before the plot finishes rendering.

A blade section makes noise when the turbulent boundary layer on its surface sweeps past the
trailing edge. Whether a given frequency is loud does not depend on that frequency in hertz —
it depends on how it compares to the natural timescale of the eddies doing the sweeping. Those
eddies are about $\delta^*$ across and they are travelling at about $U$, so their timescale is
$\delta^*/U$, and the dimensionless number that matters is

$$\mathrm{St} = \frac{f\,\delta^*}{U}$$

the Strouhal number. Two runs at the same Strouhal number are the *same experiment* as far as
the trailing edge is concerned, even if one is a 2.5 cm blade at 30 m/s and the other a 30 cm
blade at 70 m/s.

If that is right, frequency, thickness and speed are not three independent knobs. They should
mostly enter through one combination. That is a checkable claim, and it costs one plot.

In [ ]:
s2.plot_collapse(df);

On the left, frequency on its own. Twenty-one vertical stripes, because the tunnel only ever
used twenty-one band centres, and each stripe is 25 dB tall. Knowing the frequency of a
measurement tells you very little about how loud it was.

On the right, the same rows against $f\delta^*/U$. The stripes are gone, the scatter has a
shape, and there is a hump: the noise peaks somewhere around $\mathrm{St}\approx0.1$ and falls
away on both sides. That hump is the spectral shape function the acoustics literature spends
its time parameterising, and it just fell out of a scatter plot.

The number in each title is how much of the variance in the level that one variable accounts
for on its own, allowing it any shape at all. Frequency gets about a fifth. The Strouhal number
gets about half. No new measurements and no extra columns — the same three numbers, divided.

So: same search, same operators, same budget, and we swap `log_f` for `log_St`.

In [ ]:
X_st, names_st = view(df, "strouhal")
print("features:", names_st)

t = time.time()
st_model = PySRRegressor(**SEARCH)
st_model.fit(X_st[train], y[train], variable_names=names_st)
print(f"{time.time() - t:.0f}s")

st_front = front_scores(st_model, X_st[test], y[test])
best_st = best_on_front(st_front)
print()
print(best_st["equation"])
print(f"\ncomplexity {best_st['complexity']}    held-out R^2 = {best_st['test_r2']:.3f}"
      f"    (was {best_log['test_r2']:.3f})")

### What it cost, rather than what it reached

"Best on the front" is a soft way to compare searches, because it mostly reports how long you
were willing to leave things running. The sharper question is how much equation you had to buy
to reach a given accuracy. Pick a bar and ask each of the three what it charged.

In [ ]:
BAR = 0.55

fronts = {"as measured": raw_front, "logged": log_front, "Strouhal": st_front}
s2.price_of(fronts, BAR)

In [ ]:
s2.plot_fronts(
    [(k, v) for k, v in fronts.items()],
    baseline=BASELINE,
    title="Same operators, same budget, same rows — three ways of naming the inputs",
);

That is the figure the session is about, and it should be read left to right rather than top to
bottom. The two curves for the raw and logged columns sit almost on top of each other, which is
attempt two's whole story in one picture. The Strouhal curve is somewhere else entirely: it
clears the bar in noticeably fewer nodes, and it stays above the others essentially everywhere.

And now the equation is short enough to actually read.

In [ ]:
rung = shortest_above(st_front, BAR)
print(rung["equation"])
print(f"\ncomplexity {rung['complexity']}    held-out R^2 = {rung['test_r2']:.3f}")

Strip the constants off whatever came out there and you are looking at a downward parabola in
$\log \mathrm{St}$ — a level, minus something squared, with the Strouhal number and the chord
inside the square. In other words

$$\mathrm{SPL} \;\approx\; A \;-\; B\,\big[\log_{10}(\mathrm{St}\cdot c) + C\big]^2$$

which is a hump, centred where the bracket vanishes, falling off quadratically in log-frequency
on both sides. That is the shape we saw on the right-hand panel of the collapse plot, and it is
recognisably the same object the BPM model writes down as its spectral shape function — fitted
here in about a minute, from a scatter plot and one division, by a search that was never told
any acoustics.

Worth being precise about what did and did not happen. We did not add data. We did not add
operators, iterations, or population size, and we did not touch the loss. We divided three
columns we already had, because we knew what they meant. The search then spent its budget on
the shape of the curve instead of on rediscovering the division.

In [ ]:
s2.plot_parity(
    [("as measured", raw_model.predict(X_raw[test], index=int(best_raw.name))),
     ("logged", log_model.predict(X_log[test], index=int(best_log.name))),
     ("Strouhal", st_model.predict(X_st[test], index=int(best_st.name))),
     ("boosted trees", trees.predict(X_all[test]))],
    y[test],
    title="Held-out rows: predicted against measured",
);

The trees are still well ahead, and honestly they should be — four hundred of them against an
expression you can read in one breath. The Strouhal panel is visibly the tightest of the three
equations, particularly down at the quiet end where the other two lose the thread.

But the more useful thing in this figure is what all three of our equations have in common.
Look at the top right of each of the first three panels: the measurements run up to 141 dB, and
not one of the expressions will predict much above 131. They flatten. The trees do not.

Before guessing why, it costs nothing to ask the residuals.

In [ ]:
resid = df.iloc[test].assign(residual=y[test] - st_model.predict(X_st[test], index=int(best_st.name)))

print(f"loudest rows (> 132 dB), n = {(resid.spl > 132).sum()}:"
      f"  under-predicted by {resid.loc[resid.spl > 132, 'residual'].mean():.1f} dB on average\n")
for by in ["U", "chord"]:
    means = resid.groupby(by)["residual"].mean()
    print(f"  by {by:6s}  residual runs {means.min():+.1f} to {means.max():+.1f} dB")
aoa_bins = resid.groupby(pd.qcut(resid["aoa"], 4, duplicates="drop"), observed=True)["residual"].mean()
print(f"  by aoa     residual runs {aoa_bins.min():+.1f} to {aoa_bins.max():+.1f} dB")

So it is not a bias in any one variable. Split the residuals by speed, by chord or by angle of
attack and they sit within a decibel or so of zero either way, which is about as unbiased as
this data gets. The error is concentrated somewhere much more specific: on the loudest rows,
and those are the ones sitting near the top of the spectral hump.

The hump we fitted is real, but it is too flat on top. A parabola in $\log \mathrm{St}$ is a
decent description of the roll-off on either side and a poor description of the crest — so the
model shaves several decibels off exactly the measurements that a noise engineer cares about
most.

Which makes it a good thing to go after in the next twenty minutes.

---

## 6. Your turn

Everything from here is open. The scoreboard below is where answers go; re-run it whenever you
have something new.

The point of this half hour is not to win. It is to get comfortable reaching for a knob and
seeing what it does, so that the next time you have a dataset of your own you already know what
the moves are.

In [ ]:
board = [
    {"name": "boosted trees (400)", "complexity": None, "test_r2": BASELINE},
    {"name": "SR, as measured", "complexity": int(best_raw["complexity"]), "test_r2": best_raw["test_r2"]},
    {"name": "SR, logged", "complexity": int(best_log["complexity"]), "test_r2": best_log["test_r2"]},
    {"name": "SR, Strouhal", "complexity": int(best_st["complexity"]), "test_r2": best_st["test_r2"]},
]

s2.scoreboard(board)

### Things worth trying

The first three go after the flat crest that the residuals just pointed at. The rest are
general. This is an ordering, not a ranking — disagreeing with it is a perfectly good use of
the next twenty minutes.

**Give the peak a better shape.** A parabola in $\log \mathrm{St}$ is too blunt at the top. Add
an operator that can make a sharper crest and see if the search reaches for it —
`unary_operators=[..., "sqrt", "cube"]`, or a custom one such as
`"peak(x) = 1/(1 + x*x)"` with the matching `extra_sympy_mappings`.

**Weight the loud rows.** If the error is concentrated on the loudest measurements, say so in
the loss. `model.fit(X, y, weights=...)` takes a per-row weight, and weighting by something
like `(y - y.min())` tells the search where you want the accuracy spent. Be honest with
yourself about whether the overall $R^2$ then went up or you just moved the error somewhere
you were not looking.

**Give it a second dimensionless group.** The Strouhal number was one, and it was worth more
than every other move in this notebook combined. There are others here: the Mach number
$U/c_\text{sound}$, the ratio $\delta^*/\text{chord}$, the angle of attack in radians. Add one
as a column and see whether the search takes it.

**Give it more room.** `maxsize` and `niterations` are both set low so the walkthrough would fit
in the slot. Raising them is the least clever thing on this list and it is not the least
effective. Watch where the held-out curve stops rising, because that is the point where extra
complexity has stopped being physics and started being memorisation.

**Stop it nesting.** `nested_constraints={"exp": {"exp": 0, "log": 0}, "log": {"log": 0, "exp": 0}}`
— the same lever from this morning. `exp(exp(...))` is never the answer here, and every
candidate spent on one is a candidate not spent elsewhere.

**Split the problem.** The literature does not model this as one mechanism. Trailing-edge noise
from an attached boundary layer and the noise from a separated, near-stalled one are written as
separate additive terms. Try fitting the low-angle and high-angle rows separately and asking
whether the two equations are the same equation.

**Make the test harder.** Every number above comes from a random split, so the test rows sit
between the training rows. Hold out an entire wind speed instead (`df.U == 71.3`) and ask which
model degrades least. Fair warning: everything degrades, the trees included. That is itself the
interesting result, and it is the honest test of whether a fit knows any physics.

In [ ]:
# Scratch space. `SEARCH` is the config the walkthrough used; copy it and change one thing.

my_search = dict(SEARCH)
my_search["niterations"] = 80

X_mine, names_mine = view(df, "strouhal")

t = time.time()
mine = PySRRegressor(**my_search)
mine.fit(X_mine[train], y[train], variable_names=names_mine)
print(f"{time.time() - t:.0f}s")

my_front = front_scores(mine, X_mine[test], y[test])
best_mine = best_on_front(my_front)
print()
print(best_mine["equation"])
print(f"\ncomplexity {best_mine['complexity']}    held-out R^2 = {best_mine['test_r2']:.3f}")

In [ ]:
board.append({"name": "mine", "complexity": int(best_mine["complexity"]),
              "test_r2": best_mine["test_r2"]})
s2.scoreboard(board)

---

## 7. Where that leaves us

The trees are probably still on top of your scoreboard, and that is fine. Four hundred of them
against a line of algebra was never a fair fight on accuracy, and accuracy was not what this
morning was about.

What the exercise does show is where the leverage was. Two of our three moves were changes to
how the search was set up, and one was a change to what we believed about the problem. The
first two are the ones that feel like work. The third is the one that paid, and it did not
touch the search at all — it came down to dividing a frequency by a timescale, because we knew
there was a timescale.

That is the argument of the whole day in one dataset. A search will find you the best
expression it can in the space you hand it. Nearly all of the physics is in choosing that
space.